## This notebook will test using a tensor DNN to try to get similar performance to the paper

Will attempt to get a testing implementation working to try optimising parameters.

In [6]:
import os
import time
from collections.abc import Sized, Iterable

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

In [61]:

csv_path = "./HIGGS.csv"

# --- feature set -----------------------------------------------------
# 28 columns total. Columns 1-21 are the "low-level" kinematic features
# columns 22-28 are the 7 "high-level" derived features (m_jj ... m_wwbb)
feature_set = "low"               # "all" | "low"

# --- architecture ----------------------------------------------------
# Paper's best: "five-layer" net = 4 hidden x 300 + output
hidden_layers = [300, 300, 300, 300, 300]
# GELU appears best!
act_layer = nn.GELU # nn.ReLU, nn.SiLU nn.Tanh, or possibly nn.GELU
batchnorm = True                  # stabiliser see paper https://arxiv.org/abs/1502.03167 - nn.BatchNorm1d
dropout = 0.2                     # paper applied 50% dropout to the TOP hidden layer
initial_dropout = 0.0


# --- optimisation ----------------------------------------------------
epochs = 30                       # with 10M rows each epoch already sees a LOT of data
batch_size = 8192                 # big batches keep the GPU busy; paper used 100 (2011 GPU)
max_lr = 0.003                     # OneCycle peak LR for AdamW
weight_decay = 0.00001               # matches the paper's L2 coefficient
label_smoothing = 0.0             # try 0.01-0.05 if the net overfits

# --- engineering -----------------------------------------------------
amp = True                        # bfloat16 autocast (RTX 50-series supports it natively)
compile = True                    # torch.compile fuses kernels; big speedup, falls back if it fails
keep_data_on_gpu = True           # 10M x 28 float32 ~ 1.1 GB -> fits in 16 GB, avoids per-batch copies
early_stop_patience = 6           # stop if val AUC hasn't improved for this many epochs
seed = 1337

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Read and split the data

In [ ]:

def make_splits(X, y):
    n = len(X)
    print(f"Len: {n}")
    n_test, n_val = 500_000, 500_000
    tr_end = n - n_test - n_val
    return (
        X[:tr_end],      y[:tr_end],          # train  (~10M)
        X[tr_end:-n_test], y[tr_end:-n_test], # val    (500k)
        X[-n_test:],     y[-n_test:],         # test   (500k)
    )


In [ ]:

data = pd.read_csv(csv_path, header=None, dtype=np.float32)

arr = data.to_numpy()


In [52]:
# can't use the standard split method with new types
y = arr[:, 0].astype(np.int8)      # first column = label
X = arr[:, 1:].astype(np.float32)  # remaining 28 columns = features

# Select the feature subset (mirrors the paper's three experiments).
if feature_set == "low":
    X = X[:, :21]                  # 21 low-level kinematic features

X_train, X_val1, y_train, y_val1 = train_test_split(X, y, test_size=0.0909090909, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_val1, y_val1, test_size=0.5, random_state=42)

print(X_train.shape)
print(X_val.shape)
print(X_test.shape)
print(y_train.shape)
print(y_val.shape)
print(y_test.shape)


# X_train_small, X_val1_small, y_train_small, y_val1_small = train_test_split(X_small, y_small, test_size=0.0909090909, random_state=42)
# X_val_small, X_test_small, y_val_small, y_test_small = train_test_split(X_val1_small, y_val1_small, test_size=0.5, random_state=42)

(10000000, 21)
(500000, 21)
(500000, 21)
(10000000,)
(500000,)
(500000,)


#### Model class

Creates the DNN

In [50]:

class HiggsMLP(nn.Module):

    def __init__(self, in_dim: int):
        super().__init__()

        layers = []
        prev = in_dim
        n_hidden = len(hidden_layers)
        is_first = not dropout_all_layers and initial_dropout > 0
        for i, width in enumerate(hidden_layers):
            layers.append(nn.Linear(prev, width))
            if batchnorm:
                layers.append(nn.BatchNorm1d(width))
            layers.append(act_layer())
            if is_first:
                is_first = False
                layers.append(nn.Dropout(initial_dropout))
            is_last_hidden = (i == n_hidden - 1)
            if dropout > 0 and (dropout_all_layers or is_last_hidden):
                layers.append(nn.Dropout(dropout))
            prev = width

        layers.append(nn.Linear(prev, 1))    # single logit for binary classification
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)        # shape [B] to match the label vector

@torch.no_grad()
def compute_auc(model, X_gpu, y_np, batch: int = 65536):
    model.eval()
    probs = []
    for i in range(0, len(X_gpu), batch):
        logits = model(X_gpu[i:i + batch])
        probs.append(torch.sigmoid(logits).float().cpu().numpy())
    return roc_auc_score(y_np, np.concatenate(probs))

#### Train

Method to use the above class to train a single model

In [62]:
torch.manual_seed(seed)
np.random.seed(seed)
# TF32 matmuls: free accuracy-for-speed trade on NVIDIA GPUs for fp32 ops.
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# ---- data -----------------------------------------------------------

# Xtr, ytr, Xva, yva, Xte, yte = make_splits(X, y)
# Xtr, Xva, Xte = standardize(Xtr, Xva, Xte)
print(f"train={len(X_train):,}  val={len(X_val):,}  test={len(X_test):,}  features={X_train.shape[1]}")

# Move tensors to the GPU ONCE. With ~10M x 28 floats (~1.1 GB) the whole
# training set lives in the 16 GB of VRAM, so each step is a pure-GPU index
# slice -- no CPU->GPU copy per batch, which is otherwise the #1 bottleneck.
# (If your data did NOT fit in VRAM, you'd instead use a DataLoader with
#  pin_memory=True, num_workers>0, and .to(DEVICE, non_blocking=True).)
to_gpu = keep_data_on_gpu
Xtr_t = torch.tensor(X_train, device=DEVICE if to_gpu else "cpu")
ytr_t = torch.tensor(y_train, dtype=torch.float32, device=DEVICE if to_gpu else "cpu")
Xva_t = torch.tensor(X_val, device=DEVICE)
Xte_t = torch.tensor(X_test, device=DEVICE)

# ---- model / loss / optimiser --------------------------------------
model = HiggsMLP(in_dim=X_train.shape[1]).to(DEVICE)

if compile:
    try:
        model = torch.compile(model)
        print("torch.compile: enabled")
    except Exception as e:
        print(f"torch.compile failed ({e}); continuing uncompiled")

# BCEWithLogitsLoss = sigmoid + binary cross-entropy, fused & numerically stable.
loss_fn = nn.BCEWithLogitsLoss()
# AdamW is the modern default (decoupled weight decay)
opt = torch.optim.AdamW(model.parameters(), lr=max_lr, weight_decay=weight_decay)
# Basic, similar to original paper
# opt = torch.optim.SGD(model.parameters(), lr=0.05, momentum=0.9, weight_decay=1e-5)

n_train = len(Xtr_t)
steps_per_epoch = (n_train + batch_size - 1) // batch_size
# OneCycle: LR warms up then anneals within the run. Robust, fast-converging
# schedule for MLPs; a modern replacement for the paper's slow manual decay.
sched = torch.optim.lr_scheduler.OneCycleLR(
    opt, max_lr=max_lr,
    epochs=epochs, steps_per_epoch=steps_per_epoch,
)

# bfloat16 autocast: half-precision math on the GPU, ~2x throughput. bf16
# (unlike fp16) has enough dynamic range that NO GradScaler is needed.
amp_ctx = (torch.autocast("cuda", dtype=torch.bfloat16)
           if amp and DEVICE.type == "cuda"
           else torch.autocast("cuda", enabled=False))

# ---- epoch loop -----------------------------------------------------
best_auc, best_state, epochs_no_improve = 0.0, None, 0
for epoch in range(1, epochs + 1):
    model.train()
    t0 = time.perf_counter()
    perm = torch.randperm(n_train, device=Xtr_t.device)  # fresh shuffle each epoch
    running = 0.0
    for i in range(0, n_train, batch_size):
        idx = perm[i:i + batch_size]
        xb = Xtr_t[idx]
        yb = ytr_t[idx]
        if not to_gpu:                       # only needed if data is on CPU
            xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)

        opt.zero_grad(set_to_none=True)      # set_to_none is slightly faster than zeroing
        with amp_ctx:
            logits = model(xb)
            loss = loss_fn(logits, yb)
        loss.backward()
        opt.step()
        sched.step()                         # OneCycle steps PER BATCH, not per epoch
        running += loss.item() * len(idx)

    val_auc = compute_auc(model, Xva_t, y_val)
    dt = time.perf_counter() - t0
    print(f"epoch {epoch:2d}/{epochs}  loss={running / n_train:.4f}  "
          f"val_auc={val_auc:.4f}  lr={sched.get_last_lr()[0]:.2e}  ({dt:.1f}s)")

    # Early stopping: keep the best-on-validation weights, stop if stalled.
    if val_auc > best_auc + 0.0001:
        best_auc = val_auc
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= early_stop_patience:
            print(f"early stopping (no val improvement for {early_stop_patience} epochs)")
            break

# ---- final test evaluation -----------------------------------------
if best_state is not None:
    model.load_state_dict(best_state)        # restore best-validation weights
test_auc = compute_auc(model, Xte_t, y_test)
print("\n" + "=" * 60)
print(f"BEST VAL AUC : {best_auc:.4f}")
print(f"TEST AUC     : {test_auc:.4f}")
print("=" * 60)

train=10,000,000  val=500,000  test=500,000  features=21
torch.compile: enabled
epoch  1/30  loss=0.6030  val_auc=0.7660  lr=2.07e-04  (1.9s)
epoch  2/30  loss=0.5616  val_auc=0.7933  lr=4.57e-04  (1.8s)
epoch  3/30  loss=0.5340  val_auc=0.8148  lr=8.40e-04  (2.0s)
epoch  4/30  loss=0.5123  val_auc=0.8307  lr=1.31e-03  (2.0s)
epoch  5/30  loss=0.4935  val_auc=0.8449  lr=1.81e-03  (2.0s)
epoch  6/30  loss=0.4773  val_auc=0.8551  lr=2.28e-03  (1.9s)
epoch  7/30  loss=0.4659  val_auc=0.8602  lr=2.66e-03  (1.9s)
epoch  8/30  loss=0.4576  val_auc=0.8661  lr=2.91e-03  (1.9s)
epoch  9/30  loss=0.4510  val_auc=0.8692  lr=3.00e-03  (2.0s)
epoch 10/30  loss=0.4457  val_auc=0.8716  lr=2.98e-03  (1.9s)
epoch 11/30  loss=0.4413  val_auc=0.8739  lr=2.93e-03  (1.9s)
epoch 12/30  loss=0.4375  val_auc=0.8759  lr=2.85e-03  (1.9s)
epoch 13/30  loss=0.4343  val_auc=0.8769  lr=2.74e-03  (1.9s)
epoch 14/30  loss=0.4314  val_auc=0.8783  lr=2.60e-03  (1.9s)
epoch 15/30  loss=0.4287  val_auc=0.8789  lr=2.44e-0